# Concrete — TPL Heuristic Evaluation (with new MAD-based heuristic)

**8 Models** | **100 Epochs** | **4 Outlier Types × 3 Levels** | **5 Runs**

Models compared:
- Pinball, TPL-3sig, TPL-tau, **TPL-tauMAD (new)**, TPL-sweep, HPB, MAQR, QRF

α for each TPL variant is computed twice:
- **Clean-α**: from residuals of MSE-MLP trained on clean data → used for clean evaluation
- **Contam-α**: from residuals of MSE-MLP trained on each contaminated set → used for that contamination's evaluation

Plots saved to `plots/Concrete/`

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
import torch, torch.nn as nn, torch.optim as optim
from scipy.stats import norm, skewnorm
from scipy.stats import rankdata
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import LinearRegression as LR_sk
from sklearn.datasets import fetch_openml
import warnings, time, os

warnings.filterwarnings("ignore")

DATA_DIR = "."
DN = "Concrete"
PLOTS_DIR = f"plots/{DN}"
os.makedirs(PLOTS_DIR, exist_ok=True)

SPLIT_SEED = 58
SEEDS = [42, 58, 123, 256, 789]
N_RUNS = len(SEEDS)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE} | Runs: {N_RUNS} | Plots → {PLOTS_DIR}/")

QUANTILES = [
    0.01,
    0.025,
    0.03,
    0.05,
    0.09,
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.60,
    0.70,
    0.80,
    0.90,
    0.91,
    0.93,
    0.95,
    0.97,
    0.975,
    0.99,
]
OUTLIER_TYPES = ["gaussian", "multiply", "uniform", "skewed"]
CONTAMINATION_LEVELS = [0.02, 0.05, 0.10]
EP = 100
H = 100
LR = 0.01
BS = 32

MODELS = [
    "Pinball",
    "TPL-3sig",
    "TPL-tau",
    "TPL-tauMAD",
    "TPL-sweep",
    "HPB",
    "MAQR",
    "QRF",
]
COLORS = {
    "Pinball": "salmon",
    "TPL-3sig": "steelblue",
    "TPL-tau": "crimson",
    "TPL-tauMAD": "purple",
    "TPL-sweep": "seagreen",
    "HPB": "darkorange",
    "MAQR": "brown",
    "QRF": "gray",
}
sns.set_style("whitegrid")
plt.rcParams.update({"figure.figsize": (13, 5), "font.size": 11})

Device: cuda | Runs: 5 | Plots → plots/Concrete/


In [ ]:
raw = pd.read_csv(f"{DATA_DIR}/concrete.txt", header=None, sep=r"\s+", engine="python")
y_all = raw.iloc[:, -1].values.astype(float)
X_all = raw.iloc[:, :-1].values.astype(float)

X_all = StandardScaler().fit_transform(X_all)
DIM = X_all.shape[1]
Xtv, X_test, ytv, y_test_clean = train_test_split(
    X_all, y_all, test_size=0.20, random_state=SPLIT_SEED
)
X_train, X_val, y_train_clean, y_val_clean = train_test_split(
    Xtv, ytv, test_size=0.20, random_state=SPLIT_SEED
)
print(f"{DN}: d={DIM} Tr={X_train.shape[0]} Val={X_val.shape[0]} Te={X_test.shape[0]}")
print(f"Target: μ={y_all.mean():.2f} σ={y_all.std():.2f}")

Concrete: d=8 Tr=659 Val=165 Te=206
Target: μ=35.82 σ=16.70


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(14, 5))
axs[0].hist(y_all, bins=50, color="steelblue", edgecolor="white")
axs[0].set_title(f"{DN} Target Distribution")
axs[1].boxplot(y_all)
axs[1].set_title("Target Boxplot")
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/{DN}_Target_Distribution.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {DN}_Target_Distribution.png")

Saved: Concrete_Target_Distribution.png


In [ ]:
def inject_outliers(y, frac, method, seed=None):
    rng = np.random.RandomState(seed)
    y = y.copy()
    n = len(y)
    no = max(1, int(frac * n))
    ix = rng.choice(n, no, replace=False)
    mu, sig = y.mean(), y.std()
    if method == "gaussian":
        y[ix] = rng.normal(mu, np.sqrt(5) * sig, no)
    elif method == "multiply":
        y[ix] = y[ix] * rng.choice([-1, 1], no) * 10
    elif method == "uniform":
        y[ix] = rng.uniform(y.min() - 3 * sig, y.max() + 3 * sig, no)
    elif method == "skewed":
        y[ix] = skewnorm.rvs(a=10, loc=mu, scale=3 * sig, size=no, random_state=rng)
    return y


cont_sets = {}
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        cont_sets[(ot, cl)] = inject_outliers(y_train_clean, cl, ot, seed=SPLIT_SEED)
print(f"{len(cont_sets)} contaminated sets")

12 contaminated sets


In [ ]:
fig, axs = plt.subplots(2, 2, figsize=(14, 10))
for ax, ot in zip(axs.ravel(), OUTLIER_TYPES):
    yc = cont_sets[(ot, 0.10)]
    ax.hist(
        y_train_clean,
        bins=50,
        alpha=0.5,
        color="steelblue",
        label="Clean",
        density=True,
    )
    ax.hist(yc, bins=50, alpha=0.5, color="red", label="Contaminated", density=True)
    ax.set_title(f"{ot.capitalize()} contamination (10%)")
    ax.legend(fontsize=9)
plt.suptitle(
    f"{DN} — Outlier Injection (10% contamination, 4 types)", fontsize=14, y=1.01
)
plt.tight_layout()
plt.savefig(
    f"{PLOTS_DIR}/{DN}_Outlier_Injection_Visualization.png",
    dpi=120,
    bbox_inches="tight",
)
plt.close()
print(f"Saved: {DN}_Outlier_Injection_Visualization.png")

Saved: Concrete_Outlier_Injection_Visualization.png


In [ ]:
class MLP(nn.Module):
    def __init__(s, d):
        super().__init__()
        s.net = nn.Sequential(nn.Linear(d, H), nn.ReLU(), nn.Linear(H, 1))

    def forward(s, x):
        return s.net(x).squeeze(-1)


def pb_loss(p, y, tau):
    e = y - p
    return torch.mean(torch.max(tau * e, (tau - 1) * e))


def tpl_loss(p, y, tau, alpha):
    u = y - p
    return torch.mean(
        torch.where(
            (u >= 0) & (u < alpha * (1 - tau)),
            tau * u,
            torch.where(
                u >= alpha * (1 - tau),
                torch.tensor(tau * (1 - tau) * alpha, device=u.device, dtype=u.dtype),
                torch.where(
                    (u < 0) & (u >= -tau * alpha),
                    (tau - 1) * u,
                    torch.tensor(
                        -tau * (tau - 1) * alpha, device=u.device, dtype=u.dtype
                    ),
                ),
            ),
        )
    )


def hpb_loss(p, y, tau, delta=1.0):
    e = y - p
    ae = torch.abs(e)
    h = torch.where(ae <= delta, 0.5 * e**2 / delta, ae - 0.5 * delta)
    return torch.mean(torch.where(e >= 0, tau, 1 - tau) * h)


def sseed(s):
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)


def tr(X, y, lfn, ep=EP):
    m = MLP(DIM).to(DEVICE)
    o = optim.Adam(m.parameters(), lr=LR)
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    dl = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt), batch_size=BS, shuffle=True
    )
    m.train()
    for _ in range(ep):
        for xb, yb in dl:
            o.zero_grad()
            lfn(m(xb), yb).backward()
            o.step()
    m.eval()
    return m


def pr(m, X):
    m.eval()
    with torch.no_grad():
        return m(torch.tensor(X, dtype=torch.float32).to(DEVICE)).cpu().numpy()


def tr_mse(X, y, ep=EP):
    m = MLP(DIM).to(DEVICE)
    o = optim.Adam(m.parameters(), lr=LR)
    L = nn.MSELoss()
    Xt = torch.tensor(X, dtype=torch.float32).to(DEVICE)
    yt = torch.tensor(y, dtype=torch.float32).to(DEVICE)
    dl = torch.utils.data.DataLoader(
        torch.utils.data.TensorDataset(Xt, yt), batch_size=BS, shuffle=True
    )
    m.train()
    for _ in range(ep):
        for xb, yb in dl:
            o.zero_grad()
            L(m(xb), yb).backward()
            o.step()
    m.eval()
    return m


print("Models ready.")

Models ready.


In [ ]:
class MAQR:
    def __init__(s, bfn, k=50, mx=2000):
        s.bfn = bfn
        s.k = k
        s.mx = mx

    def fit(s, X, y):
        s.res = y.ravel() - s.bfn(X).ravel()
        n = len(X)
        if n > s.mx:
            ix = np.random.choice(n, s.mx, replace=False)
            Xs, rs = X[ix], s.res[ix]
        else:
            Xs, rs = X, s.res
        s.nn = NearestNeighbors(n_neighbors=min(s.k, len(Xs)))
        s.nn.fit(Xs)
        Dx, Dp, Dr = [], [], []
        for i in range(len(Xs)):
            _, idx = s.nn.kneighbors(Xs[i : i + 1])
            nr = rs[idx[0]]
            pv = rankdata(nr, "ordinal") / (len(nr) + 1)
            for j in range(len(idx[0])):
                Dx.append(Xs[i])
                Dp.append(pv[j])
                Dr.append(nr[j])
        s.lm = LR_sk()
        s.lm.fit(np.hstack([np.array(Dx), np.array(Dp).reshape(-1, 1)]), np.array(Dr))

    def predict(s, X, qs):
        bp = s.bfn(X).ravel()
        R = np.zeros((len(X), len(qs)))
        for i, t in enumerate(qs):
            R[:, i] = bp + s.lm.predict(np.hstack([X, np.full((len(X), 1), t)]))
        return R


class QRF:
    def __init__(s, rs=42):
        s.rf = RandomForestRegressor(
            n_estimators=100, random_state=rs, min_samples_leaf=5
        )

    def fit(s, X, y):
        s.rf.fit(X, y.ravel())

    def predict(s, X, tau):
        return np.percentile(
            [t.predict(X) for t in s.rf.estimators_], tau * 100, axis=0
        )


print("MAQR/QRF ready.")

MAQR/QRF ready.


In [ ]:
# Clean-data α values: computed once from CLEAN residuals.
# These are used for the clean evaluation row.
sseed(SPLIT_SEED)
mse_m = tr_mse(X_train, y_train_clean)
res_clean = y_train_clean - pr(mse_m, X_train)

RES_STD_CLEAN = np.std(res_clean)
MAD_CLEAN = np.median(np.abs(res_clean - np.median(res_clean)))
SIGMA_MAD_CLEAN = 1.4826 * MAD_CLEAN

A3_CLEAN = 3 * RES_STD_CLEAN
# TPL-tau:    α(τ) = 3 × std(res)        / min(τ, 1−τ)
# TPL-tauMAD: α(τ) = 3 × 1.4826×MAD(res) / min(τ, 1−τ)

# Sweep over clean data
SG = [5, 10, 20, 40, 60, 80, 100, 150, 200, 250, 300]
bse, ASWEEP = 1e9, 40
for a in SG:
    pv = {}
    for t in [0.05, 0.25, 0.50, 0.75, 0.95]:
        pv[t] = pr(
            tr(X_train, y_train_clean, lambda p, y, t=t, a=a: tpl_loss(p, y, t, a)),
            X_val,
        )
    e = np.mean([abs(np.mean(y_val_clean <= pv[t]) - t) for t in pv])
    if e < bse:
        bse, ASWEEP = e, a

print(f"CLEAN-DATA α values:")
print(f"  res_std={RES_STD_CLEAN:.4f}   →  α_3sig={A3_CLEAN:.4f}")
print(f"  1.4826·MAD={SIGMA_MAD_CLEAN:.4f}")
print(f"  TPL-tau:    α(τ) = 3 × {RES_STD_CLEAN:.4f} / min(τ, 1−τ)")
print(f"  TPL-tauMAD: α(τ) = 3 × {SIGMA_MAD_CLEAN:.4f} / min(τ, 1−τ)")
print(f"  α_sweep={ASWEEP}")


def ece(yt, yp, t):
    return abs(np.mean(yt <= yp) - t)

CLEAN-DATA α values:
  res_std=5.0463   →  α_3sig=15.1390
  1.4826·MAD=4.6593
  TPL-tau:    α(τ) = 3 × 5.0463 / min(τ, 1−τ)
  TPL-tauMAD: α(τ) = 3 × 4.6593 / min(τ, 1−τ)
  α_sweep=100


In [ ]:
def train_all(Xtr, ytr, Xev, a3, sigma_std, sigma_mad, asw, seed):
    """
    a3:        single α for TPL-3sig (=3·sigma_std)
    sigma_std: std(residuals) — for TPL-tau
    sigma_mad: 1.4826·MAD(residuals) — for TPL-tauMAD
    asw:       sweep α for TPL-sweep
    """
    sseed(seed)
    R = {m: {} for m in MODELS}
    for tau in QUANTILES:
        R["Pinball"][tau] = pr(tr(Xtr, ytr, lambda p, y, t=tau: pb_loss(p, y, t)), Xev)
        R["TPL-3sig"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=a3: tpl_loss(p, y, t, a)), Xev
        )
        at_std = 3 * sigma_std / min(tau, 1 - tau)
        R["TPL-tau"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=at_std: tpl_loss(p, y, t, a)), Xev
        )
        at_mad = 3 * sigma_mad / min(tau, 1 - tau)
        R["TPL-tauMAD"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=at_mad: tpl_loss(p, y, t, a)), Xev
        )
        R["TPL-sweep"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau, a=asw: tpl_loss(p, y, t, a)), Xev
        )
        R["HPB"][tau] = pr(
            tr(Xtr, ytr, lambda p, y, t=tau: hpb_loss(p, y, t, 1.0)), Xev
        )
    mm = tr_mse(Xtr, ytr)
    mq = MAQR(lambda x: pr(mm, x), k=min(50, len(Xtr) // 5), mx=2000)
    mq.fit(Xtr, ytr)
    mp = mq.predict(Xev, QUANTILES)
    for i, t in enumerate(QUANTILES):
        R["MAQR"][t] = mp[:, i]
    qrf = QRF(rs=seed)
    qrf.fit(Xtr, ytr)
    for t in QUANTILES:
        R["QRF"][t] = qrf.predict(Xev, t)
    return R

In [ ]:
Ce = {m: {t: [] for t in QUANTILES} for m in MODELS}
Oe = {
    m: {
        (o, c): {t: [] for t in QUANTILES}
        for o in OUTLIER_TYPES
        for c in CONTAMINATION_LEVELS
    }
    for m in MODELS
}
Or = {
    m: {
        (o, c): {t: [] for t in QUANTILES}
        for o in OUTLIER_TYPES
        for c in CONTAMINATION_LEVELS
    }
    for m in MODELS
}

t0 = time.time()
for ri, seed in enumerate(SEEDS):
    print(f"\nRUN {ri+1}/{N_RUNS} (seed={seed})")

    # === Clean evaluation: uses CLEAN α values ===
    cr = train_all(
        X_train,
        y_train_clean,
        X_test,
        A3_CLEAN,
        RES_STD_CLEAN,
        SIGMA_MAD_CLEAN,
        ASWEEP,
        seed,
    )
    for m in MODELS:
        for t in QUANTILES:
            Ce[m][t].append(ece(y_test_clean, cr[m][t], t))

    # === Contamination evaluation: α RECOMPUTED from contaminated residuals ===
    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            yc = cont_sets[(ot, cl)]
            sseed(seed)
            mc = tr_mse(X_train, yc)
            res_c = yc - pr(mc, X_train)
            sigma_std_c = np.std(res_c)
            sigma_mad_c = 1.4826 * np.median(np.abs(res_c - np.median(res_c)))
            a3_c = 3 * sigma_std_c
            orr = train_all(
                X_train, yc, X_test, a3_c, sigma_std_c, sigma_mad_c, ASWEEP, seed
            )
            for m in MODELS:
                for t in QUANTILES:
                    Or[m][(ot, cl)][t].append(
                        np.sqrt(mean_squared_error(cr[m][t], orr[m][t]))
                    )
                    Oe[m][(ot, cl)][t].append(ece(y_test_clean, orr[m][t], t))
    print(f"  Done ({(time.time()-t0)/60:.1f}m)")
print(f"\n✓ {N_RUNS} runs in {(time.time()-t0)/60:.1f}m")


RUN 1/5 (seed=42)
  Done (67.4m)

RUN 2/5 (seed=58)
  Done (133.8m)

RUN 3/5 (seed=123)
  Done (199.9m)

RUN 4/5 (seed=256)
  Done (266.5m)

RUN 5/5 (seed=789)
  Done (333.7m)

✓ 5 runs in 333.7m


In [ ]:
x = list(range(len(QUANTILES)))
xt = [str(q) for q in QUANTILES]
print("Plot setup ready.")

Plot setup ready.


In [ ]:
# ECE on clean data — one plot per model
for m in MODELS:
    fig, ax = plt.subplots(figsize=(13, 5))
    mc = [np.mean(Ce[m][t]) for t in QUANTILES]
    sc = [np.std(Ce[m][t]) for t in QUANTILES]
    ax.errorbar(
        x, mc, yerr=sc, fmt="o-", color=COLORS[m], lw=2, markersize=5, capsize=3
    )
    ax.set_xticks(x)
    ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
    ax.set_title(f"{DN} — Clean ECE per Quantile: {m}", fontsize=12)
    ax.set_xlabel("Quantile τ")
    ax.set_ylabel("Empirical Calibration Error")
    ax.grid(True, ls="--", alpha=0.5)
    plt.tight_layout()
    fname = f"{PLOTS_DIR}/{DN}_Clean_ECE_{m}.png"
    plt.savefig(fname, dpi=120, bbox_inches="tight")
    plt.close()
print(f"Saved {len(MODELS)} clean ECE per-model plots")

Saved 8 clean ECE per-model plots


In [ ]:
# All models overlaid — clean ECE
fig, ax = plt.subplots(figsize=(14, 6))
for m in MODELS:
    mc = [np.mean(Ce[m][t]) for t in QUANTILES]
    ax.plot(x, mc, "o-", color=COLORS[m], lw=2, markersize=4, label=m)
ax.set_xticks(x)
ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
ax.set_title(f"{DN} — Clean ECE: All Models", fontsize=12)
ax.legend(fontsize=9)
ax.set_xlabel("Quantile τ")
ax.set_ylabel("Empirical Calibration Error")
ax.grid(True, ls="--", alpha=0.5)
plt.tight_layout()
plt.savefig(f"{PLOTS_DIR}/{DN}_Clean_ECE_All_Models.png", dpi=120, bbox_inches="tight")
plt.close()
print(f"Saved: {DN}_Clean_ECE_All_Models.png")

Saved: Concrete_Clean_ECE_All_Models.png


In [ ]:
# RMSE (clean vs outlier-trained predictions) per quantile — every (model, outlier, level)
n_plots = 0
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        for m in MODELS:
            fig, ax = plt.subplots(figsize=(13, 5))
            mn = [np.mean(Or[m][(ot, cl)][t]) for t in QUANTILES]
            sd = [np.std(Or[m][(ot, cl)][t]) for t in QUANTILES]
            ax.errorbar(
                x, mn, yerr=sd, fmt="o-", color=COLORS[m], lw=2, markersize=5, capsize=3
            )
            ax.set_xticks(x)
            ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
            ax.set_title(
                f"{DN} — RMSE per Quantile: {m} ({ot.capitalize()} {int(cl*100)}% contamination)",
                fontsize=11,
            )
            ax.set_xlabel("Quantile τ")
            ax.set_ylabel("RMSE (clean vs outlier predictions)")
            ax.grid(True, ls="--", alpha=0.5)
            plt.tight_layout()
            fname = f"{PLOTS_DIR}/{DN}_RMSE_{m}_{ot.capitalize()}_{int(cl*100)}pct.png"
            plt.savefig(fname, dpi=120, bbox_inches="tight")
            plt.close()
            n_plots += 1
print(f"Saved {n_plots} per-model RMSE plots")

Saved 96 per-model RMSE plots


In [ ]:
# ECE — clean vs contaminated overlay, per (model, outlier, level)
n_plots = 0
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        for m in MODELS:
            fig, ax = plt.subplots(figsize=(13, 5))
            mc = [np.mean(Ce[m][t]) for t in QUANTILES]
            sc_ = [np.std(Ce[m][t]) for t in QUANTILES]
            mo = [np.mean(Oe[m][(ot, cl)][t]) for t in QUANTILES]
            so_ = [np.std(Oe[m][(ot, cl)][t]) for t in QUANTILES]
            ax.errorbar(
                x,
                mc,
                yerr=sc_,
                fmt="o-",
                color=COLORS[m],
                lw=2,
                markersize=5,
                capsize=3,
                label="Clean",
            )
            ax.errorbar(
                x,
                mo,
                yerr=so_,
                fmt="s--",
                color=COLORS[m],
                lw=1.5,
                markersize=5,
                capsize=3,
                alpha=0.7,
                label=f"{ot.capitalize()} {int(cl*100)}%",
            )
            ax.set_xticks(x)
            ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
            ax.set_title(
                f"{DN} — ECE Clean vs Contaminated: {m} ({ot.capitalize()} {int(cl*100)}%)",
                fontsize=11,
            )
            ax.legend()
            ax.set_xlabel("Quantile τ")
            ax.set_ylabel("Empirical Calibration Error")
            ax.grid(True, ls="--", alpha=0.5)
            plt.tight_layout()
            fname = f"{PLOTS_DIR}/{DN}_ECE_CleanVs_{m}_{ot.capitalize()}_{int(cl*100)}pct.png"
            plt.savefig(fname, dpi=120, bbox_inches="tight")
            plt.close()
            n_plots += 1
print(f"Saved {n_plots} per-model ECE clean-vs-contaminated plots")

Saved 96 per-model ECE clean-vs-contaminated plots


In [ ]:
# All models on one plot, per (outlier, level)
n_plots = 0
for ot in OUTLIER_TYPES:
    for cl in CONTAMINATION_LEVELS:
        fig, ax = plt.subplots(figsize=(14, 6))
        for m in MODELS:
            mn = [np.mean(Or[m][(ot, cl)][t]) for t in QUANTILES]
            ax.plot(x, mn, "o-", color=COLORS[m], lw=2, markersize=4, label=m)
        ax.set_xticks(x)
        ax.set_xticklabels(xt, rotation=45, ha="right", fontsize=8)
        ax.set_title(
            f"{DN} — RMSE All Models ({ot.capitalize()} {int(cl*100)}% contamination)",
            fontsize=12,
        )
        ax.legend(fontsize=9)
        ax.set_xlabel("Quantile τ")
        ax.set_ylabel("RMSE (clean vs outlier predictions)")
        ax.grid(True, ls="--", alpha=0.5)
        plt.tight_layout()
        fname = (
            f"{PLOTS_DIR}/{DN}_RMSE_All_Models_{ot.capitalize()}_{int(cl*100)}pct.png"
        )
        plt.savefig(fname, dpi=120, bbox_inches="tight")
        plt.close()
        n_plots += 1
print(f"Saved {n_plots} all-models RMSE overlay plots")

Saved 12 all-models RMSE overlay plots


In [ ]:
print(f"\nSUMMARY — {DN}")
print(
    f"Clean-α: α_3sig={A3_CLEAN:.4f} | σ_std={RES_STD_CLEAN:.4f} | σ_MAD={SIGMA_MAD_CLEAN:.4f} | α_sweep={ASWEEP}"
)
rows = []
for m in MODELS:
    r = {
        "Model": m,
        "ECE_clean": round(np.mean([np.mean(Ce[m][t]) for t in QUANTILES]), 4),
    }
    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            k = (ot, cl)
            r[f"RMSE_{ot}_{int(cl*100)}"] = round(
                np.mean([np.mean(Or[m][k][t]) for t in QUANTILES]), 4
            )
            r[f"ECE_{ot}_{int(cl*100)}"] = round(
                np.mean([np.mean(Oe[m][k][t]) for t in QUANTILES]), 4
            )
    rows.append(r)
sdf = pd.DataFrame(rows)
print(sdf.to_string(index=False))


SUMMARY — Concrete
Clean-α: α_3sig=15.1390 | σ_std=5.0463 | σ_MAD=4.6593 | α_sweep=100
     Model  ECE_clean  RMSE_gaussian_2  ECE_gaussian_2  RMSE_gaussian_5  ECE_gaussian_5  RMSE_gaussian_10  ECE_gaussian_10  RMSE_multiply_2  ECE_multiply_2  RMSE_multiply_5  ECE_multiply_5  RMSE_multiply_10  ECE_multiply_10  RMSE_uniform_2  ECE_uniform_2  RMSE_uniform_5  ECE_uniform_5  RMSE_uniform_10  ECE_uniform_10  RMSE_skewed_2  ECE_skewed_2  RMSE_skewed_5  ECE_skewed_5  RMSE_skewed_10  ECE_skewed_10
   Pinball     0.0353           4.0779          0.0331           7.2256          0.0261           12.6732           0.0231          21.6892          0.0341          54.3921          0.0286          134.3070           0.0268          6.2663         0.0353         11.3422         0.0270          15.6414          0.0259         5.4674        0.0352        11.0222        0.0331         15.9966         0.0351
  TPL-3sig     0.5177          11.2589          0.4688          16.2672          0.4139         

In [ ]:
import openpyxl

ep = f"{DN}_results.xlsx"
with pd.ExcelWriter(ep, engine="openpyxl") as w:
    # Alpha sheet — includes both std-based and MAD-based
    pd.DataFrame(
        {
            "Method": ["TPL-3sig", "TPL-tau", "TPL-tauMAD", "TPL-sweep"],
            "Alpha (clean-data)": [
                round(A3_CLEAN, 4),
                f"3×{RES_STD_CLEAN:.4f}/min(τ,1-τ)",
                f"3×{SIGMA_MAD_CLEAN:.4f}/min(τ,1-τ)  [1.4826·MAD]",
                ASWEEP,
            ],
        }
    ).to_excel(w, sheet_name="Alpha", index=False)

    # RMSE sheets per contamination
    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            k = (ot, cl)
            rows = []
            for m in MODELS:
                r = {"Model": m}
                for t in QUANTILES:
                    r[t] = round(np.mean(Or[m][k][t]), 4)
                r["Avg"] = round(np.mean([np.mean(Or[m][k][t]) for t in QUANTILES]), 4)
                rows.append(r)
            pd.DataFrame(rows).to_excel(
                w, sheet_name=f"RMSE_{ot}_{int(cl*100)}"[:31], index=False
            )

    # ECE on clean data
    rows = []
    for m in MODELS:
        r = {"Model": m}
        for t in QUANTILES:
            r[t] = round(np.mean(Ce[m][t]), 4)
        r["Avg"] = round(np.mean([np.mean(Ce[m][t]) for t in QUANTILES]), 4)
        rows.append(r)
    pd.DataFrame(rows).to_excel(w, sheet_name="ECE_Clean", index=False)

    # ECE per contamination
    for ot in OUTLIER_TYPES:
        for cl in CONTAMINATION_LEVELS:
            k = (ot, cl)
            rows = []
            for m in MODELS:
                r = {"Model": m}
                for t in QUANTILES:
                    r[t] = round(np.mean(Oe[m][k][t]), 4)
                r["Avg"] = round(np.mean([np.mean(Oe[m][k][t]) for t in QUANTILES]), 4)
                rows.append(r)
            pd.DataFrame(rows).to_excel(
                w, sheet_name=f"ECE_{ot}_{int(cl*100)}"[:31], index=False
            )

    sdf.to_excel(w, sheet_name="Summary", index=False)
print(f"✓ {ep}")

✓ Concrete_results.xlsx


## Notes — Concrete

- **TPL-3sig**: α = 3 × std(residuals) (single value)
- **TPL-tau**: α(τ) = 3 × std(residuals) / min(τ, 1−τ)
- **TPL-tauMAD (new)**: α(τ) = 3 × 1.4826·MAD(residuals) / min(τ, 1−τ)
- **TPL-sweep**: α chosen by minimizing validation ECE on clean data
- Alpha for clean evaluation: computed from clean residuals
- Alpha for each contamination condition: recomputed from that contamination's residuals (honest setup)
- 100 epochs, 5 seeds, all plots saved to `plots/Concrete/`
